# Finding the Best Time to Post: A Hacker News Platform Exploration

In this project we aim to uncover two questions regarding the Hacker News (HN) platform:
1) Do `"Ask HN"` or `"Show HN"` receive more comments on average?
2) Do posts created at a certain time receive more comments on average?

In order to answer these two questions we will take the following steps:
1) Clean the dataset
2) Extract the "Ask HN" and "Show HN" posts
3) Calculate the average number of comments for "Ask HN" and "Show HN" posts
4) Find the number of posts and comments by hour created
5) Calculate the average number of comments on posts by hour


Before we jump in, let's take a look at what Hacker News is.

[Hacker News](https://news.ycombinator.com/) is a website developed by the company, [Y Combinator](https://www.ycombinator.com/). Y Combinator is a organization that gives financial aid and mentorship to a select group of startup companies. Y Combinator created Hacker News (originally known as "Startup News") so people who were intellectually curious could communicate. The platform is similar to Reddit or Digg with conversation surrounding entrepreneurship and beyond. On the Hacker News platform, people post stories/news that is on their mind and other users can comment and/or vote on the posts. 

`"Show HN"` is the category that holds any post/project/story that has been posted while `"Ask HN"` are questions submitted on the platform. 

**Analysis Findings:** 

On average, "Ask HN" posts had more comments than "Show HN" posts. Further analysis was focused on "Ask HN" posts due to this finding. In the end, our analysis suggested that submissions at 3:00pm receive the most number of comments on average. 

For further details, see the full analysis below.

### Dataset Initiation and Exploration

The dataset being used has 20,100 rows of data and excludes all posts with no comments. [Attached](https://dq-content.s3.amazonaws.com/356/hacker_news.csv) is a link to the dataset being used in this analysis. We will use this data to read in the file.

In [1]:
# open and read the file
from csv import reader
with open("hacker_news.csv") as opened_file:
    read_file = reader(opened_file)
    HN_data = list(read_file)

# exclude header from dataset
header = HN_data[0]
HN_data = HN_data[1:]

In [2]:
# function that displays row from dataset in easy-to-read manner
def explore_data(dataset, start, end, rows_and_columns=False):
    dataset_slice = dataset[start:end]    
    for row in dataset_slice:
        print(row)
        print('\n')

    if rows_and_columns:
        print('Number of rows:', len(dataset))
        print('Number of columns:', len(dataset[0]))        

In [3]:
# exploration of first 5 rows in dataset
print(header)
print('\n')
explore_data(HN_data, 0, 5, True)

['id', 'title', 'url', 'num_points', 'num_comments', 'author', 'created_at']


['12224879', 'Interactive Dynamic Video', 'http://www.interactivedynamicvideo.com/', '386', '52', 'ne0phyte', '8/4/2016 11:52']


['10975351', 'How to Use Open Source and Shut the Fuck Up at the Same Time', 'http://hueniverse.com/2016/01/26/how-to-use-open-source-and-shut-the-fuck-up-at-the-same-time/', '39', '10', 'josep2', '1/26/2016 19:30']


['11964716', "Florida DJs May Face Felony for April Fools' Water Joke", 'http://www.thewire.com/entertainment/2013/04/florida-djs-april-fools-water-joke/63798/', '2', '1', 'vezycash', '6/23/2016 22:20']


['11919867', 'Technology ventures: From Idea to Enterprise', 'https://www.amazon.com/Technology-Ventures-Enterprise-Thomas-Byers/dp/0073523429', '3', '1', 'hswarna', '6/17/2016 0:01']


['10301696', 'Note by Note: The Making of Steinway L1037 (2007)', 'http://www.nytimes.com/2007/11/07/movies/07stein.html?_r=0', '8', '2', 'walterbell', '9/30/2015 4:12']


Number of 

This dataset has 7 columns and the first 5 rows are printed above for your review. 
A full review of the column titles in the order in which they appear in the Hacker News dataset is below:
* `id`: unique identifier
* `title`: title of the post
* `url`: URL the post links to
* `num_points`: total number of points aquired by taking the difference of the upvotes and downvotes
* `num_comments`: total number of comments on the post
* `author`: username of person who submitted the post
* `created_at`: date and time of posts creation

### Data Cleansing

The dataset we are working with in this project has been preprocessed and doesn't need extensive cleaning. For this reason, we will only check for correct column length and duplicate entries.

In [4]:
# Check that all rows have 7 column entries
for index, row in enumerate(HN_data):
    if len(row) != 7:
        print(index)
        print(element)

In [5]:
# Check for duplicates
unique_ids = []
duplicate_ids = []

for row in HN_data:
    title = row[0]
    if title in unique_ids:
        duplicate_ids.append(title)
    else:
        unique_ids.append(title)

print(len(unique_ids))
print(len(duplicate_ids))

20100
0


As anticipated, there are no rows with missing columns and no duplicate entires. Let's move on to data seperation. 

### Seperation of "Ask HN" and "Show HN" Posts

In order to analyze the distinct qualities of the different posts, we will categorize submissions based on type. 

As stated earlier, our goal is to determine whether `"Ask HN"` posts or `"Show HN"` posts receive more comments on average. So our first step will be to seperate the posts into these two categories. In the Hacker News dataset, each title that is started with "Show HN" is a post/project/story, and each title that is started with "Ask HN" is a question. In this analysis we will not be using natural language processing on titles, so any submission that do not fall into the above two categories will be added to an "Other" category. 

In [6]:
# Seperate data entries into Ask HN, Show HN, or Other based on type of post
ask_post = []
show_post = []
other_post = []

for row in HN_data:
    title = row[1]
    title = title.strip().lower()     # Convert titles to lower case and remove spaces before or after 
    if title.startswith('ask hn'):
        ask_post.append(row)
    elif title.startswith('show hn'):
        show_post.append(row)
    else:
        other_post.append(row)

In [7]:
# Function to easily display the number of posts in each category
def print_length(list_name, name):
    print(f'There are {len(list_name)} posts in {name}')
        
print_length(ask_post, 'Ask HN')
print_length(show_post, 'Show HN')
print_length(other_post, 'Other')

There are 1744 posts in Ask HN
There are 1162 posts in Show HN
There are 17194 posts in Other


Our code was able to loop through the dataset and seperate the posts into their proper categories. It appears that only a small number of items in the dataset are posts in the "Ask HN" (1,744) and "Show HN" (1,162) category while most other submissions are a different type. 

### Do "Ask HN" or "Show HN" Receive more Comments on Average?

One of the the goals for this project is to uncover:
* Do `"Ask HN"` or `"Show HN"` receive more comments on average?

We can answer this question relatively easily using the lists we created in the previous section. In our next block of code, we will loop over each list and add up the number of comments every post receives in its category. Then we will divide by the number of rows in that list to get the average. 

In [8]:
# Loop over ask_post list to count the total number of comments 
total_ask_comments = 0

for item in ask_post:
    num_comments = int(item[4])
    total_ask_comments += num_comments

# Take the average
avg_ask_comments = total_ask_comments/(len(ask_post))    
print(f'ASK_HN average comments: {avg_ask_comments}')    

ASK_HN average comments: 14.038417431192661


In [9]:
# Loop over show_post list to count the total number of comments
total_show_comments = 0

for item in show_post:
    num_comments = int(item[4])
    total_show_comments += num_comments

# Take the average
avg_show_comments = total_show_comments/(len(show_post))
print(f'SHOW_HN average comments: {avg_show_comments}')

SHOW_HN average comments: 10.31669535283993


Remember...
* `"Ask HN"` is the category that holds any *questions* that are submitted

* `"Show HN"` is the category that holds any *post/project/story* submissions

Conclusion:

On average, "Ask HN" has about 14 comments and "Show HN" has about 10 comments. So asking a question on the Hacker News platform will likely get you more replies than posting a story, but only by a slim margin. Since "Ask HN" submissions receive more comments on average, we will focus our remaining analysis on that category of posts. 

### Do Posts Created at a Certain Time Receive more Comments on Average?

The second question we aim to answer is: 
* Do posts created at a certain time receive more comments on average? 

Our inital analysis will focus on "Ask HN" submissions and we will follow these steps to answer the question:
1) Find the number of "Ask HN" posts and comments by hour created
2) Calculate the average number of comments for "Ask HN" posts by hour

#### Find the Number of "Ask HN" Posts and Comments by Hour Created

From our dataset we will need to collect every occurance of:
1) The date when a post was created *and*
2) The number of comments it received

Luckily, we already have a list of lists with only "Ask HN" entries (`ask_post`) so obtaining a new list with those two values will not be challenging. After we collect those values into one list, we will create two dictionaries: 
1) One dictionary will hold the number of posts created by hour *and*
2) The second dictionary will hold the total number of comments based on the hour the post was created

In [10]:
import datetime as dt

post_date_list = []    # List of lists with values: (1) time when post was created and (2) the number of comments it received

for row in ask_post:
    created_at = row[6]
    num_comments = int(row[4])
    post_date_list.append([created_at, num_comments])


from pprint import pprint

day_hr_counts = {}     # Number of posts created per hour
day_hr_comments = {}   # Total comments based on hour when post was created

for row in post_date_list:
    date = row[0]
    comments = row[1]
    datetime_object = dt.datetime.strptime(date, '%m/%d/%Y %H:%M')
    hour_of_day = dt.datetime.strftime(datetime_object, '%H')
    if hour_of_day in day_hr_counts:
        day_hr_counts[hour_of_day] += 1
        day_hr_comments[hour_of_day] += comments
    else:
        day_hr_counts[hour_of_day] = 1
        day_hr_comments[hour_of_day] = comments

In [11]:
print('Number of posts created by hour:')
pprint(day_hr_counts)

Number of posts created by hour:
{'00': 55,
 '01': 60,
 '02': 58,
 '03': 54,
 '04': 47,
 '05': 46,
 '06': 44,
 '07': 34,
 '08': 48,
 '09': 45,
 '10': 59,
 '11': 58,
 '12': 73,
 '13': 85,
 '14': 107,
 '15': 116,
 '16': 108,
 '17': 100,
 '18': 109,
 '19': 110,
 '20': 80,
 '21': 109,
 '22': 71,
 '23': 68}


Notice that time is formatted as a 24-hr clock so '00' is midnight (12:00am) and '12' is noon (12:00pm). We will discuss the results in the standard, 12-hr clock format.

From this data is appears that most posts are made from 2:00pm to 8:00pm with a small peak at 3:00pm. Post submissions pick up around 12:00pm and tapper off around 9:00pm.

In [12]:
print('Total number of commented based on hour when posted was created:')
pprint(day_hr_comments)

Total number of commented based on hour when posted was created:
{'00': 447,
 '01': 683,
 '02': 1381,
 '03': 421,
 '04': 337,
 '05': 464,
 '06': 397,
 '07': 267,
 '08': 492,
 '09': 251,
 '10': 793,
 '11': 641,
 '12': 687,
 '13': 1253,
 '14': 1416,
 '15': 4477,
 '16': 1814,
 '17': 1146,
 '18': 1439,
 '19': 1188,
 '20': 1722,
 '21': 1745,
 '22': 479,
 '23': 543}


Most comments are made on submissions that posted from the hours of 1:00pm to 9:00pm with a peak at 3:00pm. An exception to this trend is surge in comments on posts submitted at 2:00am. If we will notice, there is an identical peak at 3:00pm in both datasets. 

An important distinction to note is that the `day_hr_comments` is *not* the time of day when most people comment. These values only tell us about the number of comments a post receives based on what hour of day *the post was submitted*. So, based on the results, a post at 3:00pm will likely receive the most number of comments. 

**Conclusion:**

The most common time to post is at 3:00pm with submissions at 3:00pm simultaneously receiving the most amount of comments. The most common range of time to post is between 2:00pm and 7:00pm with submissions between 1:00pm and 9:00pm receiving the most number of comments. 

#### Calculating the Average Number of Comments for "Ask HN" Posts by Hour

While our above analysis outputs totals, it does not necessarily give us an accurate picture of when a post is more likely to receive the highest number of comments. Since the above analysis only counts comments, our totals may be skewed if one post at a certain hour had an abnormally large number of comments compared to the remaining posts. To account for this, we will analyze the average number of comments for every post at a given hour. There are other, more precise statistical methods to account for outliers, but a simple average will do for this project. 

In [13]:
# List of average number of comments a post received based on the hour it was posted
avg_by_hour = []

for hour in day_hr_comments:
    avg = round((day_hr_comments[hour]/day_hr_counts[hour]), 2)
    avg_by_hour.append([hour, avg])

# key=lambda to direct 'sorted' func to sort results using index 1 of lists in avg_by_hour
avg_sort = sorted(avg_by_hour, key=lambda a_list:a_list[1], reverse=True) 
pprint(avg_sort)

[['15', 38.59],
 ['02', 23.81],
 ['20', 21.52],
 ['16', 16.8],
 ['21', 16.01],
 ['13', 14.74],
 ['10', 13.44],
 ['14', 13.23],
 ['18', 13.2],
 ['17', 11.46],
 ['01', 11.38],
 ['11', 11.05],
 ['19', 10.8],
 ['08', 10.25],
 ['05', 10.09],
 ['12', 9.41],
 ['06', 9.02],
 ['00', 8.13],
 ['23', 7.99],
 ['07', 7.85],
 ['03', 7.8],
 ['04', 7.17],
 ['22', 6.75],
 ['09', 5.58]]


In [14]:
print('Top 5 Hours for Ask Posts Comments:')

# loop through top 5 entries in avg_sort and format cleanly
for row in avg_sort[:5]:
    hour = row[0]
    avg_com = row[1]
    hour_object = dt.datetime.strptime(hour, '%H')
    hour_format = hour_object.strftime('%H:%M')
    print(f'{hour_format}: {avg_com}')

Top 5 Hours for Ask Posts Comments:
15:00: 38.59
02:00: 23.81
20:00: 21.52
16:00: 16.8
21:00: 16.01


**Conclusion:**

On average, submissions at 3:00pm receive the most number of comments. 

2:00am posts received the second highest average number of comments and posts at 8:00pm receive the third most. Something of note, if we look at the submission times, 2:00am and 8:00pm, they did not necessarily have a proportionally large number of posts to large number of comments. In both cases the number of posts were proportionally low to the number of comments.

**Further Evaluation Ideas:**
* Determine if show or ask posts receive more points on average
* Determine if posts created at a certain time are more likely to receive more points